In [1]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

import gc
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer


# Configuration

random_state = 42
batch_size = 32
max_seq_length = 256
warmup_size = 32

model_registry = {
    "minilm": {
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "expected_dimension": 384,
    },
    "mpnet": {
        "model_id": "sentence-transformers/all-mpnet-base-v2",
        "expected_dimension": 768,
    },
    "gte_modernbert": {
        "model_id": "Alibaba-NLP/gte-modernbert-base",
        "expected_dimension": 768,
    },
}

project_directory = Path(
    "/content/drive/MyDrive/movie_embeddings_project"
)

embeddings_directory = (
    project_directory
    / "artifacts"
    / "embeddings"
)

metrics_directory = (
    project_directory
    / "artifacts"
    / "metrics"
)

metadata_directory = (
    project_directory
    / "artifacts"
    / "metadata"
)

embeddings_directory.mkdir(
    parents=True,
    exist_ok=True,
)

metrics_directory.mkdir(
    parents=True,
    exist_ok=True,
)

metadata_directory.mkdir(
    parents=True,
    exist_ok=True,
)


# Reproducibility and GPU validation

np.random.seed(random_state)
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

assert torch.cuda.is_available(), (
    "GPU nije dostupan. U Colab-u izaberi "
    "Runtime > Change runtime type > T4 GPU."
)

device = "cuda"
gpu_name = torch.cuda.get_device_name(0)

print(f"GPU: {gpu_name}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")


# Locate movies_cleaned.csv

possible_dataset_paths = [
    Path("/content/movies_cleaned.csv"),
    Path("/content/drive/MyDrive/movies_cleaned.csv"),
    project_directory / "movies_cleaned.csv",
    (
        project_directory
        / "data"
        / "processed"
        / "movies_cleaned.csv"
    ),
]

dataset_path = next(
    (
        path
        for path in possible_dataset_paths
        if path.exists()
    ),
    None,
)

if dataset_path is None:
    drive_matches = list(
        Path("/content/drive/MyDrive").rglob(
            "movies_cleaned.csv"
        )
    )

    if not drive_matches:
        raise FileNotFoundError(
            "movies_cleaned.csv nije pronađen. "
            "Uploaduj ga u Colab ili Google Drive."
        )

    dataset_path = drive_matches[0]

print(f"Dataset: {dataset_path}")


# Load and validate dataset

movies_df = pd.read_csv(dataset_path)

required_columns = {
    "movie_id",
    "embedding_text",
}

missing_columns = (
    required_columns - set(movies_df.columns)
)

assert not missing_columns, (
    f"Nedostaju kolone: {sorted(missing_columns)}"
)

assert movies_df["movie_id"].notna().all()
assert movies_df["movie_id"].is_unique
assert movies_df["embedding_text"].notna().all()

texts = (
    movies_df["embedding_text"]
    .astype(str)
    .tolist()
)

movie_ids = (
    movies_df["movie_id"]
    .to_numpy(dtype=np.int64)
)

print(f"Broj filmova: {len(movies_df)}")
print(
    "Jedinstveni movie_id: "
    f"{movies_df['movie_id'].nunique()}"
)

dataset_hash = hashlib.sha256(
    dataset_path.read_bytes()
).hexdigest()

movie_ids_path = (
    metadata_directory
    / "embedding_movie_ids.npy"
)

np.save(
    movie_ids_path,
    movie_ids,
)


# Benchmark

benchmark_results = []

for model_name, config in model_registry.items():
    print("\n" + "=" * 70)
    print(f"Model: {model_name}")
    print(f"Repository: {config['model_id']}")
    print("=" * 70)

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    # Model download je već završen,
    # pa load_time ne uključuje download.

    load_start = time.perf_counter()

    model = SentenceTransformer(
        config["model_id"],
        device=device,
    )

    model.max_seq_length = max_seq_length

    torch.cuda.synchronize()

    load_time = (
        time.perf_counter() - load_start
    )

    # Warm-up se ne računa u encode vreme.

    _ = model.encode(
        texts[:warmup_size],
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    encode_start = time.perf_counter()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    torch.cuda.synchronize()

    encode_time = (
        time.perf_counter() - encode_start
    )

    peak_gpu_memory_mb = (
        torch.cuda.max_memory_allocated()
        / 1024**2
    )

    # Standardizujemo dtype radi fer poređenja
    # fajlova i stabilnije numeričke analize.

    embeddings = embeddings.astype(
        np.float32,
        copy=False,
    )

    expected_dimension = (
        config["expected_dimension"]
    )

    assert embeddings.shape == (
        len(movies_df),
        expected_dimension,
    ), (
        f"Neočekivan shape za {model_name}: "
        f"{embeddings.shape}"
    )

    assert np.isfinite(embeddings).all(), (
        f"{model_name} sadrži NaN ili inf vrednosti."
    )

    # Ponovna normalizacija je potrebna zato što
    # GTE može vratiti float16 embeddinge.

    embedding_norms = np.linalg.norm(
        embeddings,
        axis=1,
        keepdims=True,
    )

    assert np.all(embedding_norms > 0), (
        f"{model_name} sadrži embedding sa normom 0."
    )

    embeddings = (
        embeddings / embedding_norms
    )

    embedding_norms = np.linalg.norm(
        embeddings,
        axis=1,
    )

    assert np.allclose(
        embedding_norms,
        1.0,
        atol=1e-6,
    ), (
        f"{model_name} embeddingi nisu "
        "L2-normalizovani."
    )

    output_path = (
        embeddings_directory
        / f"{model_name}_embeddings.npy"
    )

    np.save(
        output_path,
        embeddings,
    )

    file_size_mb = (
        output_path.stat().st_size
        / 1024**2
    )

    movies_per_second = (
        len(texts) / encode_time
    )

    metadata = {
        "model_name": model_name,
        "model_id": config["model_id"],
        "dataset_path": str(dataset_path),
        "dataset_sha256": dataset_hash,
        "number_of_movies": len(movies_df),
        "embedding_dimension": embeddings.shape[1],
        "dtype": str(embeddings.dtype),
        "batch_size": batch_size,
        "max_seq_length": max_seq_length,
        "normalize_embeddings": True,
        "device": device,
        "gpu_name": gpu_name,
        "movie_ids_file": str(
            movie_ids_path
        ),
    }

    metadata_path = (
        metadata_directory
        / f"{model_name}_metadata.json"
    )

    with open(
        metadata_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            indent=2,
        )

    benchmark_results.append(
        {
            "model": model_name,
            "model_id": config["model_id"],
            "hardware": gpu_name,
            "number_of_movies": len(texts),
            "embedding_dimension": embeddings.shape[1],
            "dtype": str(embeddings.dtype),
            "batch_size": batch_size,
            "max_seq_length": max_seq_length,
            "load_time_seconds": load_time,
            "encode_time_seconds": encode_time,
            "movies_per_second": movies_per_second,
            "peak_gpu_memory_mb": peak_gpu_memory_mb,
            "embedding_file_mb": file_size_mb,
            "mean_l2_norm": float(
                embedding_norms.mean()
            ),
            "min_l2_norm": float(
                embedding_norms.min()
            ),
            "max_l2_norm": float(
                embedding_norms.max()
            ),
        }
    )

    print(f"Shape: {embeddings.shape}")
    print(f"Dtype: {embeddings.dtype}")
    print(f"Load vreme: {load_time:.2f} s")
    print(f"Encode vreme: {encode_time:.2f} s")
    print(
        "Brzina: "
        f"{movies_per_second:.2f} filmova/s"
    )
    print(
        "Peak GPU memorija: "
        f"{peak_gpu_memory_mb:.2f} MB"
    )
    print(
        f"Veličina fajla: {file_size_mb:.2f} MB"
    )
    print(f"Sačuvano: {output_path}")

    del model
    del embeddings
    del embedding_norms

    gc.collect()
    torch.cuda.empty_cache()


# Final benchmark table

benchmark_df = pd.DataFrame(
    benchmark_results
)

benchmark_df = (
    benchmark_df
    .sort_values("encode_time_seconds")
    .reset_index(drop=True)
)

benchmark_path = (
    metrics_directory
    / "model_benchmark_t4.csv"
)

benchmark_df.to_csv(
    benchmark_path,
    index=False,
)

display(
    benchmark_df[
        [
            "model",
            "hardware",
            "embedding_dimension",
            "dtype",
            "encode_time_seconds",
            "movies_per_second",
            "peak_gpu_memory_mb",
            "embedding_file_mb",
        ]
    ]
)

print(
    f"\nBenchmark sačuvan u:\n{benchmark_path}"
)

print(
    f"\nEmbeddingi sačuvani u:\n"
    f"{embeddings_directory}"
)

print(
    f"\nMetadata sačuvan u:\n"
    f"{metadata_directory}"
)

GPU: Tesla T4
PyTorch: 2.11.0+cu128
CUDA: 12.8
Dataset: /content/movies_cleaned.csv
Broj filmova: 9967
Jedinstveni movie_id: 9967

Model: minilm
Repository: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/312 [00:00<?, ?it/s]

Shape: (9967, 768)
Dtype: float32
Load vreme: 7.41 s
Encode vreme: 21.82 s
Brzina: 456.72 filmova/s
Peak GPU memorija: 471.05 MB
Veličina fajla: 29.20 MB
Sačuvano: /content/drive/MyDrive/movie_embeddings_project/artifacts/embeddings/gte_modernbert_embeddings.npy


,model,hardware,embedding_dimension,dtype,encode_time_seconds,movies_per_second,peak_gpu_memory_mb,embedding_file_mb
0,minilm,Tesla T4,384,float32,7.795920,1278.489308,213.440918,14.600220
1,gte_modernbert,Tesla T4,768,float32,21.822967,456.720655,471.046387,29.200317
2,mpnet,Tesla T4,768,float32,46.353971,215.019336,802.586914,29.200317



Benchmark sačuvan u:
/content/drive/MyDrive/movie_embeddings_project/artifacts/metrics/model_benchmark_t4.csv

Embeddingi sačuvani u:
/content/drive/MyDrive/movie_embeddings_project/artifacts/embeddings

Metadata sačuvan u:
/content/drive/MyDrive/movie_embeddings_project/artifacts/metadata


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


DATA_PATH = Path("data/processed/movies_cleaned.csv")
EMBEDDINGS_DIR = Path("artifacts/embeddings")

EMBEDDING_PATHS = {
    "minilm": EMBEDDINGS_DIR / "minilm_embeddings.npy",
    "mpnet": EMBEDDINGS_DIR / "mpnet_embeddings.npy",
    "gte_modernbert": EMBEDDINGS_DIR / "gte_modernbert_embeddings.npy",
}


# Load dataset
movies_df = pd.read_csv(DATA_PATH)

# Load embedding matrices
embeddings = {
    model_name: np.load(file_path)
    for model_name, file_path in EMBEDDING_PATHS.items()
}

print(f"Number of movies: {len(movies_df):,}")
print(f"Dataset shape: {movies_df.shape}")
print()


validation_rows = []

for model_name, matrix in embeddings.items():
    if matrix.ndim != 2:
        raise ValueError(
            f"{model_name}: expected a 2D matrix, got shape {matrix.shape}"
        )

    row_count_matches = matrix.shape[0] == len(movies_df)
    nan_count = int(np.isnan(matrix).sum())
    inf_count = int(np.isinf(matrix).sum())

    l2_norms = np.linalg.norm(matrix, axis=1)

    validation_rows.append(
        {
            "model": model_name,
            "shape": str(matrix.shape),
            "num_movies": matrix.shape[0],
            "embedding_dimension": matrix.shape[1],
            "dtype": str(matrix.dtype),
            "row_count_matches": row_count_matches,
            "nan_count": nan_count,
            "inf_count": inf_count,
            "mean_l2_norm": l2_norms.mean(),
            "std_l2_norm": l2_norms.std(),
            "min_l2_norm": l2_norms.min(),
            "max_l2_norm": l2_norms.max(),
        }
    )

    if not row_count_matches:
        raise ValueError(
            f"{model_name}: {matrix.shape[0]} embeddings, "
            f"but dataset contains {len(movies_df)} movies."
        )

    if nan_count > 0:
        raise ValueError(f"{model_name}: found {nan_count} NaN values.")

    if inf_count > 0:
        raise ValueError(f"{model_name}: found {inf_count} infinite values.")


embedding_validation_df = pd.DataFrame(validation_rows)

embedding_validation_df

Number of movies: 9,967
Dataset shape: (9967, 12)



,model,shape,num_movies,embedding_dimension,dtype,row_count_matches,nan_count,inf_count,mean_l2_norm,std_l2_norm,min_l2_norm,max_l2_norm
0,minilm,"(9967, 384)",9967,384,float32,True,0,0,1.0,4.135931e-08,1.0,1.0
1,mpnet,"(9967, 768)",9967,768,float32,True,0,0,1.0,4.590942e-08,1.0,1.0
2,gte_modernbert,"(9967, 768)",9967,768,float32,True,0,0,1.0,3.983597e-08,1.0,1.0
